# 33장 · StateGraph 기초 · 상태, 노드, 간선을 연결해 실행하기

StateGraph는 상태를 읽고 일부 값을 갱신하는 함수를 연결합니다. add_node로 함수를 등록하고 add_edge로 순서를 정한 뒤, compile한 그래프에 초기 상태를 넣으면 됩니다.

## 노드 한 개가 있는 그래프 실행하기

`add_node`에 이름과 함수를 등록하고, `add_edge`로 실행 순서를 연결합니다. `compile()`로 만든 그래프의 `invoke`에 시작값을 넣어 보세요. 아래에서는 노드 하나가 2를 읽어 3을 반환합니다. 새 커널에서는 `%pip install "langgraph>=1,<2"`로 준비하세요.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class Count(TypedDict):
    value: int

def increment(state):
    return {"value": state["value"] + 1}

builder = StateGraph(Count)
builder.add_node("plus_one", increment)
builder.add_edge(START, "plus_one")
builder.add_edge("plus_one", END)
graph = builder.compile()
print(graph.invoke({"value": 2}))

{'value': 3}


plus_one은 간선에서 쓸 이름이고 increment는 실행할 함수입니다. invoke가 value=2를 넘기면 함수가 3을 반환하고 그래프가 종료됩니다.

## 응용 실습 · 두 노드로 문장 길이 판정하기

짝 활동에는 [실습 수첩](pre-mid/documents/graph/workshop.md)을 사용합니다. 과제 아래에서 풀이를 펼칠 수 있습니다.

## TypedDict로 상태의 필드 정하기

그래프의 상태는 노드가 읽고 갱신할 값의 묶음입니다. `TypedDict`로 각 키의 이름과 자료형을 적습니다. 처음에는 문장만 넣고, 길이를 구한 노드는 새로 계산한 필드만 반환합니다. 다음 노드는 갱신된 상태를 받습니다.

교재 ZIP을 풀고 그 폴더에서 Jupyter를 엽니다. Python 3.12 커널에서 필요한 패키지가 없으면 새 셀에서 `%pip install "langgraph>=1,<2" openai streamlit pypdf python-dotenv`를 실행하고 커널을 다시 시작하세요. 실습 자료 폴더와 노트북은 함께 보관합니다.

그래프의 상태는 여러 함수가 차례로 읽고 갱신하는 값입니다. `NoticeState`는 딕셔너리에 어떤 키가 들어갈 수 있는지 나타냅니다. `total=False`라서 처음부터 모든 키를 채울 필요가 없습니다. 다만 없는 키를 `state["count"]`로 읽으면 일반 딕셔너리와 마찬가지로 KeyError가 납니다. 먼저 글자 수를 만들어 주는 노드가 필요한 이유입니다.

In [1]:
from typing import TypedDict
from langgraph.graph import START, END, StateGraph

class NoticeState(TypedDict, total=False):
    text: str
    count: int
    label: str

def count_chars(state):
    # 원문은 그대로 두고 새로 셈한 칸만 돌려줍니다.
    return {"count": len(state["text"])}

sample = {"text": "창문을 닫아 주세요"}
print(count_chars(sample))
print(sample)

{'count': 10}
{'text': '창문을 닫아 주세요'}


함수를 직접 호출한 결과에는 count만 있고 원래 딕셔너리에는 text만 있습니다. 그래프가 노드 반환값을 상태에 반영하는 단계는 아직 실행하지 않았습니다. 공백도 한 글자로 셌습니다.

## 두 번째 노드의 입력을 손으로 만들기

전광판은 공백을 포함해 열 글자까지 표시합니다. 둘째 작업자는 문장을 다시 세지 않고 첫 작업자가 남긴 숫자를 읽습니다. count가 없는 시작 상태로 이 함수를 먼저 부르면 어떤 이름에서 멈출지 예측하세요.

In [2]:
def decide_label(state):
    return {"label": "표시 가능" if state["count"] <= 10 else "줄여 주세요"}

# 첫 노드가 돌려준 칸을 더하면 둘째 노드의 입력이 됩니다.
after_count = {**sample, **count_chars(sample)}
print(after_count)
print(decide_label(after_count))

{'text': '창문을 닫아 주세요', 'count': 10}
{'label': '표시 가능'}


text와 count가 함께 있어도 두 번째 함수는 count만 읽습니다. 각 함수의 반환에 text를 계속 복사할 필요가 없습니다. 빈 문자열도 길이 0으로 통과하므로 ‘길이 통과’는 내용이 유효하다는 판단과는 구별해야 합니다.

## add_node, add_edge, compile로 그래프 조립하기

START에서 첫 노드로 들어가고, 둘째 노드가 결과를 저장하면 END에서 멈춥니다. compile은 이 연결을 실행할 수 있는 그래프로 조립합니다. invoke에 넘기는 값이 그 실행의 시작 상태입니다.

`StateGraph(NoticeState)`로 상태의 형식을 정합니다. `add_node("count_chars", count_chars)`에서 따옴표가 있는 이름은 간선에서 찾을 주소이고, 두 번째 인자는 나중에 실행할 함수입니다. 여기서 `count_chars()`처럼 괄호를 붙이면 상태를 받기 전에 실행해 버립니다. `add_edge(출발, 도착)`로 순서를 정하고 `compile()`로 실행 가능한 그래프를 얻습니다. 실제 입력을 넣는 줄은 맨 아래의 `notice_graph.invoke(sample)`입니다.

### 상태를 갱신하는 노드 하나만 호출하기

In [ ]:
def measure_text(state):
    return {"count": len(state["text"])}

before = {"text": "서문 이용"}
change = measure_text(before)
print(change)
print(before)

{'count': 5}
{'text': '서문 이용'}


공백까지 다섯 글자입니다. 함수가 반환한 값에는 count만 있고, 처음 넘긴 사전에는 여전히 text만 있습니다. 아래 그래프가 노드를 실행하면 상태에 이 변경값을 합쳐 다음 노드에 보냅니다.

In [3]:
def build_notice():
    plan = StateGraph(NoticeState)
    plan.add_node("count_chars", count_chars)
    plan.add_node("decide_label", decide_label)
    plan.add_edge(START, "count_chars")
    plan.add_edge("count_chars", "decide_label")
    plan.add_edge("decide_label", END)
    return plan.compile()

notice_graph = build_notice()
print(notice_graph.invoke(sample))

{'text': '창문을 닫아 주세요', 'count': 10, 'label': '표시 가능'}


마지막 딕셔너리에 원고가 남고 두 칸이 더 생겼습니다. 출력문을 노드 안에 넣지 않아도 호출자가 완성된 상태를 받을 수 있습니다.

## stream_mode="updates"로 노드의 반환값 하나씩 읽기

`notice_graph.invoke(...)`는 실행을 마친 상태를 반환합니다. `notice_graph.stream(..., stream_mode="updates")`를 순회하면 노드가 실행될 때마다 그 노드 이름과 이번에 반환한 필드를 읽습니다. 아래 출력에서 길이를 쓴 노드와 판정한 노드를 찾아 보세요.

In [4]:
for update in notice_graph.stream({"text": "창문과 출입문을 닫아 주세요"}, stream_mode="updates"):
    print(update)

{'count_chars': {'count': 15}}
{'decide_label': {'label': '줄여 주세요'}}


첫 줄의 수와 둘째 줄의 판정을 연결해 읽으세요. stream도 그래프를 한 번 실행합니다. 모델을 쓰는 다음 장에서는 같은 질문을 invoke한 뒤 다시 stream하면 두 번 요청할 수 있습니다.

## 앞 단계가 아직 쓰지 않은 칸을 읽으면

작업 순서가 뒤집힌 상황을 재현합니다. 시작 상태에는 text만 있으니 둘째 노드가 무엇을 찾지 못할까요? 오류를 일부러 만든 뒤 정확한 예외 이름을 확인하면, 나중에 큰 그래프에서도 모델과 무관한 상태 연결 오류를 따로 찾을 수 있습니다.

In [5]:
try:
    decide_label({"text": "오늘 정상 운영"})
except KeyError as error:
    # 일부러 빠뜨린 상태 필드 이름을 학생이 확인합니다.
    print("준비되지 않은 칸", error.args[0])

준비되지 않은 칸 count


여기서 count 기본값을 0으로 채우면 프로그램은 지나가지만 긴 문장도 표시 가능으로 판정합니다. 입력에 count를 잘못 넣은 문제인지, 연결 순서가 뒤집힌 문제인지 나누기 위해 첫 노드가 실제로 반환한 작은 딕셔너리를 봅니다.

## 실습 · 문장의 경계값 시험

담당자는 “정확히 10자도 게시할 수 있어요”라고 합니다. 글자 수 9·10·11인 원고를 학생이 만들고, 실행 전에 세 표시 상태를 적으세요. 결과가 다르면 원문에 공백이 있는지 확인합니다.

In [6]:
lengths = [9, 10, 11]
# 각 길이만큼 "가"를 반복해 notice_graph.invoke의 text에 넣고 count와 label을 출력하세요.

<details><summary>모범 예시와 풀이</summary><pre><code>for text in [&quot;가&quot; * 9, &quot;가&quot; * 10, &quot;가&quot; * 11]:
    result = notice_graph.invoke({&quot;text&quot;: text})
    print(result[&quot;count&quot;], result[&quot;label&quot;])</code></pre><p>9자와 10자는 표시 가능, 11자는 줄여 달라는 결과입니다. 조건을 &lt; 10으로 바꾸면 딱 10자인 입력에서만 결과가 바뀝니다.</p></details>

## 실습 · 남은 자리도 편집자에게 전달하기

전광판 편집자가 남은 칸 수를 요청했습니다. 기존 세 칸을 보존하며 remaining 칸을 추가하세요. 11자라면 빈 자리는 0입니다. 다른 원고를 쓴 짝에게 최종 상태와 노드별 반환을 함께 보여 주세요.

In [7]:
test_text = "가" * 11
# BoardState에 remaining을 선언하세요. 남은 칸 수를 돌려주는 노드를 마지막에 연결하고 test_text로 실행하세요.

<details><summary>모범 예시와 풀이</summary><pre><code>class BoardState(NoticeState, total=False):
    remaining: int

def remaining_space(state):
    return {&quot;remaining&quot;: max(0, 10 - state[&quot;count&quot;])}

board = StateGraph(BoardState)
board.add_sequence([count_chars, decide_label, remaining_space])
board.add_edge(START, &quot;count_chars&quot;)
board.add_edge(&quot;remaining_space&quot;, END)
compiled_board = board.compile()
print(compiled_board.invoke({&quot;text&quot;: &quot;가&quot; * 11}))</code></pre><p>새 칸을 선언하고 글자 수를 읽는 마지막 작업을 연결했습니다. 11글자일 때 count는 11, remaining은 0입니다. add_sequence는 나열한 함수 사이의 간선을 순서대로 연결합니다.</p></details>

## 전이 과제 · 접수 담당자가 빈 원고를 보냈을 때

전광판에 “게시 가능”이 떴는데 실제로 보이는 글자는 하나도 없습니다. 어제 구현한 길이 규칙에는 결함이 없고, 오늘 빈 원고도 받는다는 요구가 추가됐습니다. 이 사례에서 원고 길이와 원고의 유효성을 한 칸에 섞어 두면 “길이 초과”와 “내용 없음”이 같은 안내로 나갈 수 있습니다. 담당자는 두 상황에서 다른 행동을 해야 합니다.

원고가 빈 문자열이거나 공백만 있으면 “원고를 입력하세요”라고 안내합니다. 문장의 앞뒤 공백을 길이에서 빼도 되는지는 담당자가 아직 결정하지 않았습니다. 기존처럼 공백까지 길이에 세는 규칙을 지키면서 ‘내용이 보이는지’만 별도로 판단하세요. 빈 원고, 공백 세 개, “동문 개방”을 시험하고 첫 결과에서 담당자에게 할 질문을 적어 보세요.

In [8]:
drafts = ["", "   ", "동문 개방"]
# count_chars로 길이를 구하세요. strip()으로 빈 원고를 가려내고 원래 길이와 안내 문구를 출력하세요.

<details><summary>모범 예시와 풀이</summary><pre><code>def review_notice(state):
    if not state[&quot;text&quot;].strip():
        return {&quot;label&quot;: &quot;원고를 입력하세요&quot;}
    return decide_label(state)

for text in [&quot;&quot;, &quot;   &quot;, &quot;동문 개방&quot;]:
    draft = {&quot;text&quot;: text}
    counted = {**draft, **count_chars(draft)}
    print(counted[&quot;count&quot;], review_notice(counted)[&quot;label&quot;])</code></pre><p>세 길이는 0, 3, 5입니다. 첫 두 결과는 원고를 입력하라는 안내, 마지막 결과는 표시 가능입니다. 공백을 제거한 새 문장을 상태에 쓰면 길이 기준까지 바뀌므로 이 예시에서는 내용을 검사할 때만 strip을 씁니다. 그래프에서 decide_label 대신 review_notice를 등록해 같은 입력으로 한 번 더 검사하세요.</p></details>

다음 장에서는 길이가 넘친 원고에만 수정 기회를 줍니다. [LangGraph Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api)